In [1]:
!pip install -U deepeval
!pip install ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 812.5/812.5 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 4.8 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.37.0
    Uninstalling opentelemetry-proto-1.37.0:
      Successfully uninstalled opentelemetry-proto-1.37.0

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%%bash
sudo apt-get update && sudo apt-get install zstd -y
curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,302 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,870 kB]
Get:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease [24.3 kB]
Get:13 http://security.ubuntu.com/ubuntu

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:114

In [4]:

# Create a persistent models directory in Google Drive
!mkdir -p /content/drive/MyDrive/RAG/ollama_models

# Remove any existing ephemeral models directory
!rm -rf ~/.ollama/models

# Symlink Ollama's model directory to Google Drive
!ln -s /content/drive/MyDrive/RAG/ollama_models ~/.ollama/models

print('Symlinked ~/.ollama/models -> /content/drive/MyDrive/RAG/ollama_models')

import subprocess, time

server = subprocess.Popen(['ollama', 'serve'])
print('Ollama server started as background process (PID:', server.pid, ')')
time.sleep(5)
print('Waited 5 seconds for the server to initialize.')



ln: failed to create symbolic link '/root/.ollama/models': No such file or directory
Symlinked ~/.ollama/models -> /content/drive/MyDrive/RAG/ollama_models
Ollama server started as background process (PID: 3595 )
Waited 5 seconds for the server to initialize.


In [5]:
# LLM
!ollama pull gpt-oss:20b

#!ollama pull nomic-embed-text

In [6]:
import ast
import pandas as pd
from tqdm.auto import tqdm
import os

# Increase the timeout for DeepEval operations
os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "300"

from deepeval.models import OllamaModel
from deepeval.test_case import LLMTestCase
from deepeval.metrics import (
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    FaithfulnessMetric,
)


In [7]:
judge_llm = OllamaModel(
    model="gpt-oss:20b",
    base_url="http://localhost:11434",
    temperature=0,
    generation_kwargs={"options": {"num_ctx": 128000}},
)


In [8]:
PATH_IN = r"/content/drive/MyDrive/RAG/results/eval_result_combined_20260118.xlsx"
SHEET = "output"

df = pd.read_excel(PATH_IN, sheet_name=SHEET)

QUESTION_COL = "user_query"
GROUND_TRUTH_COL = "ref_answer"


In [9]:
import ast
import re
import pandas as pd

def safe_str(x) -> str:
    if x is None:
        return ""
    if isinstance(x, float) and pd.isna(x):
        return ""
    return str(x)

# Removes a trailing page marker line such as:
# "...schools.\n3" or "...schools.\n  12"
_page_marker_re = re.compile(r"\n\s*\d+\s*$")

# Used to split "plain string" contexts that contain repeated blocks separated by blank lines
# This splits on >= 1 blank line (i.e., two newlines with optional whitespace between)
_blank_block_split_re = re.compile(r"\n\s*\n+")

def clean_chunk(text: str) -> str:
    t = safe_str(text)
    t = t.replace("\r\n", "\n").replace("\r", "\n")
    t = _page_marker_re.sub("", t)  # drop trailing page marker
    # normalize excessive whitespace (optional; keep newlines but remove trailing spaces)
    t = "\n".join(line.rstrip() for line in t.split("\n"))
    return t.strip()

def _split_plain_string_into_chunks(s: str):
    """
    Split a plain string into chunk candidates.
    Strategy:
    - If it contains blank lines, treat each block as a chunk.
    - Otherwise treat whole string as one chunk.
    """
    s = s.strip()
    if not s:
        return []
    parts = [p.strip() for p in _blank_block_split_re.split(s) if p.strip()]
    return parts if len(parts) > 1 else [s]

def parse_retrieved_context(
    x,
    *,
    top_k=6,
    max_chars_per_chunk=2500,
    dedupe=True
):
    """
    Returns list[str] for DeepEval retrieval_context.

    Handles:
    - list objects
    - stringified python lists
    - long plain strings containing multiple blocks separated by blank lines

    Applies:
    - cleaning (strip, normalize newlines, remove trailing page marker)
    - truncation per chunk
    - deduplication (exact match after cleaning)
    - top-k selection
    """
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []

    items = None

    # Case A: already a list
    if isinstance(x, list):
        items = x

    # Case B/C: it is a string (could be stringified list OR plain blocks)
    elif isinstance(x, str):
        s = x.strip()
        if not s:
            return []

        # Try stringified list first
        if s.startswith("[") and s.endswith("]"):
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, list):
                    items = parsed
                else:
                    items = _split_plain_string_into_chunks(s)
            except Exception:
                # Not a valid literal list; treat as plain multi-block string
                items = _split_plain_string_into_chunks(s)
        else:
            # Plain multi-block string
            items = _split_plain_string_into_chunks(s)

    # Case D: other types -> coerce to string chunk
    else:
        items = [x]

    # Clean + truncate + drop empties
    cleaned = []
    for it in items:
        c = clean_chunk(it)
        if not c:
            continue
        if max_chars_per_chunk and len(c) > max_chars_per_chunk:
            c = c[:max_chars_per_chunk].rstrip() + "…"
        cleaned.append(c)

    # Dedupe repeated chunks (your sample repeats the same one many times)
    if dedupe:
        seen = set()
        uniq = []
        for c in cleaned:
            if c not in seen:
                seen.add(c)
                uniq.append(c)
        cleaned = uniq

    # Top-k
    if top_k and len(cleaned) > top_k:
        cleaned = cleaned[:top_k]

    return cleaned


In [10]:
%%time
precision_metric = ContextualPrecisionMetric(model=judge_llm, include_reason=True)
recall_metric = ContextualRecallMetric(model=judge_llm, include_reason=True)
faithfulness_metric = FaithfulnessMetric(model=judge_llm, include_reason=True)


CPU times: user 42 µs, sys: 4 µs, total: 46 µs
Wall time: 49.6 µs


In [ ]:
%%time
SYSTEMS = ["l1", "g1", "d1", "l2", "g2", "d2"]

# output columns
for sys in SYSTEMS:
    df[f"{sys}_contextual_precision_score"] = None
    df[f"{sys}_contextual_precision_reason"] = None

    df[f"{sys}_contextual_recall_score"] = None
    df[f"{sys}_contextual_recall_reason"] = None

    df[f"{sys}_faithfulness_score"] = None
    df[f"{sys}_faithfulness_reason"] = None


for sys in SYSTEMS:
    answer_col = f"{sys}_response"
    rc_col = f"{sys}_retrieved_context"

    if answer_col not in df.columns or rc_col not in df.columns:
        print(f"[SKIP] Missing columns for {sys}: {answer_col} or {rc_col}")
        continue

    print(f"Evaluating system: {sys}")

    for i, row in tqdm(df.iterrows(), total=len(df), desc=sys, leave=True):
        question = safe_str(row.get(QUESTION_COL))
        expected = safe_str(row.get(GROUND_TRUTH_COL))
        answer = safe_str(row.get(answer_col))

        # Tailored to your format (stringified list with duplicates)
        rc = parse_retrieved_context(
            row.get(rc_col),
            top_k=6,
            max_chars_per_chunk=2500,
            dedupe=True,
        )

        # Basic missing checks
        if not question.strip():
            df.at[i, f"{sys}_contextual_precision_reason"] = "Missing question"
            df.at[i, f"{sys}_contextual_recall_reason"] = "Missing question"
            df.at[i, f"{sys}_faithfulness_reason"] = "Missing question"
            continue

        # Contextual Precision
        try:
            tc = LLMTestCase(
                input=question,
                actual_output=answer,
                expected_output=expected,
                retrieval_context=rc,
            )
            score = precision_metric.measure(tc)
            df.at[i, f"{sys}_contextual_precision_score"] = score
            df.at[i, f"{sys}_contextual_precision_reason"] = precision_metric.reason
        except Exception as e:
            df.at[i, f"{sys}_contextual_precision_score"] = None
            df.at[i, f"{sys}_contextual_precision_reason"] = f"ERROR: {type(e).__name__}: {e}"

        # Contextual Recall
        try:
            tc = LLMTestCase(
                input=question,
                actual_output=answer,
                expected_output=expected,
                retrieval_context=rc,
            )
            score = recall_metric.measure(tc)
            df.at[i, f"{sys}_contextual_recall_score"] = score
            df.at[i, f"{sys}_contextual_recall_reason"] = recall_metric.reason
        except Exception as e:
            df.at[i, f"{sys}_contextual_recall_score"] = None
            df.at[i, f"{sys}_contextual_recall_reason"] = f"ERROR: {type(e).__name__}: {e}"

        # Faithfulness
        try:
            tc = LLMTestCase(
                input=question,
                actual_output=answer,
                retrieval_context=rc,
            )
            score = faithfulness_metric.measure(tc)
            df.at[i, f"{sys}_faithfulness_score"] = score
            df.at[i, f"{sys}_faithfulness_reason"] = faithfulness_metric.reason
        except Exception as e:
            df.at[i, f"{sys}_faithfulness_score"] = None
            df.at[i, f"{sys}_faithfulness_reason"] = f"ERROR: {type(e).__name__}: {e}"


✨ You're running DeepEval's latest Contextual Recall Metric! (using gpt-oss:20b (Ollama), strict=False, async_mod…

In [ ]:
PATH_OUT = r"/content/drive/MyDrive/RAG/results/eval_r2_all_systems_deepeval_cpr_cr_fa_20260118.xlsx"
df.to_excel(PATH_OUT, index=False)
print("Saved:", PATH_OUT)


In [ ]:
cols_to_show = [
    "user_query",
    "ref_answer",
    "g1_response",
    "g1_contextual_precision_score",
    "g1_contextual_recall_score",
    "g1_faithfulness_score",
]
display(df[cols_to_show].head(3))


In [ ]:
sys = "g1"
sample_idx = 0  # change if you want another row
rc_sample = parse_retrieved_context(df.loc[sample_idx, f"{sys}_retrieved_context"])

print("Parsed retrieval_context chunks:", len(rc_sample))
print("First chunk preview:\n", rc_sample[0][:600])
